# 05 — ALS Hyperparameter Tuning

This notebook tunes the **normal/full-training-data implicit ALS model** before we move to recent-only ALS and time-decayed ALS.

Main purpose:

1. Train multiple ALS configurations using the processed H&M train data.
2. Evaluate them fairly on the validation warm-start users.
3. Compare tuned ALS against the best popularity baseline.
4. Save a clean leaderboard and the best ALS configuration.
5. Produce a clear decision for the next notebook: time-decayed ALS.

Recommended Kaggle inputs:

- Processed data from `01-hm-preprocessing-fixed.ipynb`
- Popularity baseline artifacts from `03-popularity-baseline-fixed.ipynb`

This notebook intentionally avoids tuning on the test split. The test split should remain for final reporting.


In [ ]:
import os
import sys
import json
import math
import time
import gc
import pickle
import platform
import subprocess
from pathlib import Path
from typing import Dict, Iterable, List, Tuple, Optional

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, save_npz

from IPython.display import display

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 200)

print("Python:", platform.python_version())
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)


## 1. Install / import ALS dependency

Kaggle sometimes does not have `implicit` pre-installed. Keep internet on for this notebook.


In [ ]:
try:
    from implicit.als import AlternatingLeastSquares
    import implicit
    print("implicit:", implicit.__version__)
except Exception:
    print("implicit is not installed. Installing now...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "implicit", "-q"])
    from implicit.als import AlternatingLeastSquares
    import implicit
    print("implicit:", implicit.__version__)


## 2. Environment and path detection

The path logic searches both `/kaggle/working` and `/kaggle/input`, so it should work whether you run this immediately after previous notebooks or attach saved Kaggle datasets.


In [ ]:
def detect_environment() -> str:
    if (
        os.environ.get("KAGGLE_KERNEL_RUN_TYPE") is not None
        or Path("/kaggle/input").exists()
        or Path("/kaggle/working").exists()
    ):
        return "kaggle"

    if os.environ.get("COLAB_RELEASE_TAG") is not None:
        return "colab"

    return "local"


ENVIRONMENT = detect_environment()
IN_COLAB = ENVIRONMENT == "colab"
IN_KAGGLE = ENVIRONMENT == "kaggle"

print("Detected environment:", ENVIRONMENT)

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
else:
    print("Skipping Google Drive mount because this is not real Colab.")


In [ ]:
PROJECT_NAME = "hm-recommender"


def find_file(filename: str, search_roots: List[Path]) -> Optional[Path]:
    for root in search_roots:
        if not root.exists():
            continue
        direct = root / filename
        if direct.exists():
            return direct
        matches = sorted(root.rglob(filename))
        if matches:
            return matches[0]
    return None


def find_processed_data_dir() -> Path:
    expected_file = "train_interactions_aggregated.parquet"

    candidate_dirs = [
        Path("/kaggle/working") / PROJECT_NAME / "data" / "processed",
        Path("/content/drive/MyDrive") / PROJECT_NAME / "data" / "processed",
        Path.cwd() / "data" / "processed",
        Path.cwd() / PROJECT_NAME / "data" / "processed",
    ]

    for candidate in candidate_dirs:
        if (candidate / expected_file).exists():
            return candidate

    search_roots = [Path("/kaggle/input"), Path.cwd()]
    found = find_file(expected_file, search_roots)
    if found is not None:
        return found.parent

    raise FileNotFoundError(
        "Could not find processed data folder. Run 01-hm-preprocessing-fixed.ipynb first "
        "or attach its output dataset."
    )


def find_baseline_artifacts_dir() -> Optional[Path]:
    expected_candidates = [
        "popularity_baseline_metrics.csv",
        "global_popularity_scores.parquet",
        "best_popularity_model_info.json",
    ]

    candidate_dirs = [
        Path("/kaggle/working") / PROJECT_NAME / "artifacts" / "popularity_baseline",
        Path("/content/drive/MyDrive") / PROJECT_NAME / "artifacts" / "popularity_baseline",
        Path.cwd() / "artifacts" / "popularity_baseline",
        Path.cwd() / PROJECT_NAME / "artifacts" / "popularity_baseline",
    ]

    for candidate in candidate_dirs:
        if any((candidate / filename).exists() for filename in expected_candidates):
            return candidate

    search_roots = [Path("/kaggle/input"), Path.cwd()]
    for filename in expected_candidates:
        found = find_file(filename, search_roots)
        if found is not None:
            return found.parent

    return None


PROCESSED_DATA_DIR = find_processed_data_dir()
BASELINE_ARTIFACTS_DIR = find_baseline_artifacts_dir()

if IN_KAGGLE:
    PROJECT_DIR = Path("/kaggle/working") / PROJECT_NAME
elif IN_COLAB:
    PROJECT_DIR = Path("/content/drive/MyDrive") / PROJECT_NAME
else:
    PROJECT_DIR = Path.cwd()

ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
TUNING_ARTIFACTS_DIR = ARTIFACTS_DIR / "als_hyperparameter_tuning"
BEST_MODEL_DIR = TUNING_ARTIFACTS_DIR / "best_model"

for folder in [ARTIFACTS_DIR, TUNING_ARTIFACTS_DIR, BEST_MODEL_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("PROCESSED_DATA_DIR:", PROCESSED_DATA_DIR)
print("BASELINE_ARTIFACTS_DIR:", BASELINE_ARTIFACTS_DIR)
print("TUNING_ARTIFACTS_DIR:", TUNING_ARTIFACTS_DIR)


In [ ]:
required_processed_files = [
    "train_interactions.parquet",
    "val_interactions.parquet",
    "train_interactions_aggregated.parquet",
    "articles_processed.parquet",
    "article_id_map.parquet",
    "customer_id_map.parquet",
    "sample_submission_processed.parquet",
]

missing = []
for filename in required_processed_files:
    path = PROCESSED_DATA_DIR / filename
    if path.exists():
        print(f"Found {filename}: {path.stat().st_size / (1024 ** 2):.2f} MB")
    else:
        missing.append(str(path))

if missing:
    raise FileNotFoundError("Missing processed files:\n" + "\n".join(missing))

print("All required processed files found.")


## 3. Tuning configuration

Default tuning plan:

- Factors: `64`, `128`
- Alpha: `20`, `40`, `80`
- Regularization: `0.01`, `0.05`, `0.1`
- Iterations: `15`

That gives 18 main ALS models.

The notebook also has an optional focused `256`-factor stage. Instead of running every `256` combination, it takes the best alpha/regularization combinations from the main grid and tests them with 256 factors.

For speed, validation evaluation uses a fixed warm-start user sample by default. This keeps comparison fair because every model is evaluated on the same users.


In [ ]:
TOP_K_VALUES = [12, 20]
PRIMARY_K = 20
RANDOM_SEED = 42

# Tuning grid. This is the main stage.
MAIN_FACTORS = [64, 128]
ALPHA_VALUES = [20.0, 40.0, 80.0]
REGULARIZATION_VALUES = [0.01, 0.05, 0.1]
ITERATION_VALUES = [15]

# Optional focused stronger-factor stage.
RUN_FOCUSED_256_STAGE = True
N_TOP_COMBOS_FOR_256 = 3
FOCUSED_256_FACTORS = 256

# Keep this as 50,000 for first tuning. Set None only when you want full validation evaluation.
MAX_EVAL_USERS_PER_SPLIT = 50_000

# ALS recommend batch size. Reduce this to 1024 or 2048 if Kaggle memory becomes tight.
RECOMMEND_BATCH_SIZE = 4096

# Save only the best model factors, not every model. This avoids huge output size.
SAVE_BEST_MODEL_FACTORS = True
SAVE_BEST_MODEL_PICKLE = False

# This notebook should tune on validation only. Do not tune on test.
EVAL_SPLIT_NAME = "validation"

np.random.seed(RANDOM_SEED)

print("Main grid size:", len(MAIN_FACTORS) * len(ALPHA_VALUES) * len(REGULARIZATION_VALUES) * len(ITERATION_VALUES))
print("Focused 256 stage:", RUN_FOCUSED_256_STAGE)
print("MAX_EVAL_USERS_PER_SPLIT:", MAX_EVAL_USERS_PER_SPLIT)


## 4. Utility functions


In [ ]:
def memory_usage_mb(df: pd.DataFrame) -> float:
    return df.memory_usage(deep=True).sum() / (1024 ** 2)


def print_df_info(df: pd.DataFrame, name: str, show_head: bool = True) -> None:
    print()
    print(name)
    print("-" * 90)
    print("Shape:", df.shape)
    print(f"Memory usage: {memory_usage_mb(df):.2f} MB")
    if show_head:
        display(df.head())


def save_json(obj: dict, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        json.dump(obj, f, indent=2, default=str)
    print(f"Saved: {path}")


def save_csv(df: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)
    print(f"Saved: {path} | {path.stat().st_size / (1024 ** 2):.2f} MB")


def save_parquet(df: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(path, index=False, compression="snappy")
    print(f"Saved: {path} | {path.stat().st_size / (1024 ** 2):.2f} MB")


def safe_pickle(obj, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    try:
        with open(path, "wb") as f:
            pickle.dump(obj, f)
        print(f"Saved: {path} | {path.stat().st_size / (1024 ** 2):.2f} MB")
    except Exception as e:
        print("Warning: pickle save failed:", repr(e))


In [ ]:
def apk(recommended: List[int], actual: set, k: int) -> float:
    if not actual:
        return 0.0

    score = 0.0
    hits = 0
    seen_recommended = set()

    for rank, item in enumerate(recommended[:k], start=1):
        if item in seen_recommended:
            continue
        seen_recommended.add(item)
        if item in actual:
            hits += 1
            score += hits / rank

    return score / min(len(actual), k)


def ndcgk(recommended: List[int], actual: set, k: int) -> float:
    if not actual:
        return 0.0

    dcg = 0.0
    for rank, item in enumerate(recommended[:k], start=1):
        if item in actual:
            dcg += 1.0 / math.log2(rank + 1)

    ideal_hits = min(len(actual), k)
    idcg = sum(1.0 / math.log2(rank + 1) for rank in range(1, ideal_hits + 1))
    return dcg / idcg if idcg > 0 else 0.0


def precisionk(recommended: List[int], actual: set, k: int) -> float:
    if k <= 0:
        return 0.0
    hits = sum(1 for item in recommended[:k] if item in actual)
    return hits / k


def recallk(recommended: List[int], actual: set, k: int) -> float:
    if not actual:
        return 0.0
    hits = sum(1 for item in recommended[:k] if item in actual)
    return hits / len(actual)


def mrrk(recommended: List[int], actual: set, k: int) -> float:
    if not actual:
        return 0.0
    for rank, item in enumerate(recommended[:k], start=1):
        if item in actual:
            return 1.0 / rank
    return 0.0


def hitratek(recommended: List[int], actual: set, k: int) -> float:
    return float(any(item in actual for item in recommended[:k]))


In [ ]:
def build_user_item_sets(df: pd.DataFrame) -> Dict[int, set]:
    grouped = df.groupby("customer_idx", sort=False)["article_idx"].agg(
        lambda values: set(values.to_numpy(dtype=np.int32))
    )
    return grouped.to_dict()


def build_global_popularity_from_train(train_df: pd.DataFrame) -> np.ndarray:
    scores = (
        train_df
        .groupby("article_idx", as_index=False)
        .agg(
            popularity_score=("weight", "sum"),
            purchase_count=("article_idx", "size"),
            unique_customers=("customer_idx", "nunique"),
            last_train_purchase_date=("t_dat", "max"),
        )
        .sort_values(
            ["popularity_score", "unique_customers", "last_train_purchase_date", "article_idx"],
            ascending=[False, False, False, True],
        )
    )
    return scores["article_idx"].to_numpy(dtype=np.int32)


def sample_eval_users(user_ids: List[int], max_users: Optional[int], seed: int = RANDOM_SEED) -> List[int]:
    if max_users is None or len(user_ids) <= max_users:
        return list(user_ids)

    rng = np.random.default_rng(seed)
    sampled = rng.choice(np.array(user_ids, dtype=np.int32), size=max_users, replace=False)
    return sampled.astype(int).tolist()


def update_metric_sums(
    recommended: List[int],
    actual: set,
    k_values: List[int],
    sums: Dict[int, Dict[str, float]],
    unique_recommended_items_by_k: Dict[int, set],
) -> None:
    for k in k_values:
        top_k_recommended = recommended[:k]
        unique_recommended_items_by_k[k].update(top_k_recommended)
        sums[k]["precision"] += precisionk(recommended, actual, k)
        sums[k]["recall"] += recallk(recommended, actual, k)
        sums[k]["map"] += apk(recommended, actual, k)
        sums[k]["ndcg"] += ndcgk(recommended, actual, k)
        sums[k]["mrr"] += mrrk(recommended, actual, k)
        sums[k]["hitrate"] += hitratek(recommended, actual, k)


## 5. Load processed data and baseline artifacts


In [ ]:
train_interactions = pd.read_parquet(
    PROCESSED_DATA_DIR / "train_interactions.parquet",
    columns=["t_dat", "customer_idx", "article_idx", "price", "sales_channel_id", "weight"],
)

val_interactions = pd.read_parquet(
    PROCESSED_DATA_DIR / "val_interactions.parquet",
    columns=["t_dat", "customer_idx", "article_idx", "price", "sales_channel_id", "weight"],
)

train_interactions_aggregated = pd.read_parquet(
    PROCESSED_DATA_DIR / "train_interactions_aggregated.parquet",
    columns=["customer_idx", "article_idx", "purchase_count", "implicit_weight"],
)

article_id_map = pd.read_parquet(PROCESSED_DATA_DIR / "article_id_map.parquet")
customer_id_map = pd.read_parquet(PROCESSED_DATA_DIR / "customer_id_map.parquet")
articles = pd.read_parquet(PROCESSED_DATA_DIR / "articles_processed.parquet")
sample_submission = pd.read_parquet(PROCESSED_DATA_DIR / "sample_submission_processed.parquet")

for df in [train_interactions, val_interactions]:
    df["t_dat"] = pd.to_datetime(df["t_dat"])
    df["customer_idx"] = df["customer_idx"].astype("int32")
    df["article_idx"] = df["article_idx"].astype("int32")

train_interactions_aggregated["customer_idx"] = train_interactions_aggregated["customer_idx"].astype("int32")
train_interactions_aggregated["article_idx"] = train_interactions_aggregated["article_idx"].astype("int32")
train_interactions_aggregated["implicit_weight"] = train_interactions_aggregated["implicit_weight"].astype("float32")

print_df_info(train_interactions, "Train interactions")
print_df_info(train_interactions_aggregated, "Train aggregated interactions")
print_df_info(val_interactions, "Validation interactions")
print_df_info(article_id_map, "Article ID map")
print_df_info(customer_id_map, "Customer ID map")


In [ ]:
# Load popularity fallback and baseline metrics if available.
baseline_metrics = None
best_popularity_row = None

if BASELINE_ARTIFACTS_DIR is not None and (BASELINE_ARTIFACTS_DIR / "global_popularity_scores.parquet").exists():
    global_popularity_scores = pd.read_parquet(BASELINE_ARTIFACTS_DIR / "global_popularity_scores.parquet")
    global_ranked_items = global_popularity_scores["article_idx"].to_numpy(dtype=np.int32)
    print("Loaded global popularity fallback from:", BASELINE_ARTIFACTS_DIR / "global_popularity_scores.parquet")
else:
    global_ranked_items = build_global_popularity_from_train(train_interactions)
    print("Built global popularity fallback from train interactions because baseline artifact was not found.")

if BASELINE_ARTIFACTS_DIR is not None and (BASELINE_ARTIFACTS_DIR / "popularity_baseline_metrics.csv").exists():
    baseline_metrics = pd.read_csv(BASELINE_ARTIFACTS_DIR / "popularity_baseline_metrics.csv")
    baseline_primary = baseline_metrics[
        (baseline_metrics["eval_split"] == "validation")
        & (baseline_metrics["warm_start_only"] == True)
        & (baseline_metrics["k"] == PRIMARY_K)
    ].copy()

    if len(baseline_primary) > 0:
        best_popularity_row = (
            baseline_primary
            .sort_values(["recall_at_k", "ndcg_at_k", "mrr_at_k", "map_at_k"], ascending=False)
            .iloc[0]
            .to_dict()
        )
        print("Best popularity baseline for comparison:")
        display(pd.DataFrame([best_popularity_row]))
    else:
        print("Baseline metrics loaded, but no validation/warm/k=20 row found.")
else:
    print("Popularity baseline metrics not found. Tuning will continue without automatic baseline comparison.")

print("Fallback ranked items:", len(global_ranked_items))


## 6. Build sparse train matrix and validation warm-start sample


In [ ]:
n_users = int(customer_id_map["customer_idx"].max()) + 1
n_items = int(article_id_map["article_idx"].max()) + 1

row_indices = train_interactions_aggregated["customer_idx"].to_numpy(dtype=np.int32)
col_indices = train_interactions_aggregated["article_idx"].to_numpy(dtype=np.int32)
values = train_interactions_aggregated["implicit_weight"].to_numpy(dtype=np.float32)

user_items = csr_matrix(
    (values, (row_indices, col_indices)),
    shape=(n_users, n_items),
    dtype=np.float32,
)
user_items.eliminate_zeros()

print("user_items shape:", user_items.shape)
print("user_items non-zero entries:", user_items.nnz)
print("density percent:", user_items.nnz / (user_items.shape[0] * user_items.shape[1]) * 100)

save_npz(TUNING_ARTIFACTS_DIR / "train_user_items_matrix_for_tuning.npz", user_items)
print("Saved sparse matrix for audit:", TUNING_ARTIFACTS_DIR / "train_user_items_matrix_for_tuning.npz")


In [ ]:
print("Building train seen and validation target dictionaries...")
started = time.time()
train_seen_by_user = build_user_item_sets(train_interactions)
val_actual_by_user = build_user_item_sets(val_interactions)
print(f"Built dictionaries in {time.time() - started:.2f} seconds.")

val_warm_user_ids_all = [int(u) for u in val_actual_by_user.keys() if int(u) in train_seen_by_user]
val_cold_user_count = sum(1 for u in val_actual_by_user.keys() if int(u) not in train_seen_by_user)

tuning_eval_user_ids = sample_eval_users(
    user_ids=val_warm_user_ids_all,
    max_users=MAX_EVAL_USERS_PER_SPLIT,
    seed=RANDOM_SEED,
)

np.save(TUNING_ARTIFACTS_DIR / "tuning_eval_user_ids.npy", np.array(tuning_eval_user_ids, dtype=np.int32))

split_audit = {
    "train_seen_users": int(len(train_seen_by_user)),
    "validation_target_users": int(len(val_actual_by_user)),
    "validation_warm_users_all": int(len(val_warm_user_ids_all)),
    "validation_cold_users": int(val_cold_user_count),
    "tuning_eval_users_used": int(len(tuning_eval_user_ids)),
    "max_eval_users_per_split": MAX_EVAL_USERS_PER_SPLIT,
}

print(json.dumps(split_audit, indent=2))
save_json(split_audit, TUNING_ARTIFACTS_DIR / "tuning_split_audit.json")


## 7. ALS training and evaluation functions


In [ ]:
def make_model_name(config: dict) -> str:
    return (
        f"als_f{int(config['factors'])}"
        f"_reg{str(config['regularization']).replace('.', 'p')}"
        f"_it{int(config['iterations'])}"
        f"_alpha{str(config['alpha']).replace('.', 'p')}"
    )


def build_main_grid_configs() -> List[dict]:
    configs = []
    for factors in MAIN_FACTORS:
        for alpha in ALPHA_VALUES:
            for regularization in REGULARIZATION_VALUES:
                for iterations in ITERATION_VALUES:
                    config = {
                        "stage": "main_grid",
                        "factors": int(factors),
                        "alpha": float(alpha),
                        "regularization": float(regularization),
                        "iterations": int(iterations),
                    }
                    config["model_name"] = make_model_name(config)
                    configs.append(config)
    return configs


MAIN_GRID_CONFIGS = build_main_grid_configs()
print("Main grid configs:", len(MAIN_GRID_CONFIGS))
for config in MAIN_GRID_CONFIGS:
    print(config)


In [ ]:
def train_als_model(config: dict, user_items_matrix: csr_matrix) -> AlternatingLeastSquares:
    print()
    print("=" * 120)
    print("Training ALS model:", config["model_name"])
    print(config)

    model = AlternatingLeastSquares(
        factors=int(config["factors"]),
        regularization=float(config["regularization"]),
        iterations=int(config["iterations"]),
        alpha=float(config["alpha"]),
        random_state=RANDOM_SEED,
        use_gpu=False,
        calculate_training_loss=False,
    )

    started = time.time()
    model.fit(user_items_matrix, show_progress=True)
    elapsed = time.time() - started
    print(f"Training completed in {elapsed:.2f} seconds.")

    return model, elapsed


def batch_recommend_als(
    model: AlternatingLeastSquares,
    user_items_matrix: csr_matrix,
    user_ids: List[int],
    k: int,
    batch_size: int = 4096,
) -> Iterable[Tuple[int, List[int]]]:
    for start in range(0, len(user_ids), batch_size):
        batch_users = np.array(user_ids[start:start + batch_size], dtype=np.int32)
        if len(batch_users) == 0:
            continue

        try:
            ids, scores = model.recommend(
                batch_users,
                user_items_matrix[batch_users],
                N=k,
                filter_already_liked_items=True,
                recalculate_user=False,
            )
            for user_id, rec_items in zip(batch_users, ids):
                yield int(user_id), [int(item) for item in rec_items[:k] if int(item) >= 0]
        except Exception as batch_error:
            print("Batch recommend failed once; falling back to per-user recommendation.")
            print("Batch error:", repr(batch_error))
            for user_id in batch_users:
                rec_items, scores = model.recommend(
                    int(user_id),
                    user_items_matrix[int(user_id)],
                    N=k,
                    filter_already_liked_items=True,
                    recalculate_user=False,
                )
                yield int(user_id), [int(item) for item in rec_items[:k] if int(item) >= 0]


In [ ]:
def evaluate_als_on_fixed_validation_sample(
    model: AlternatingLeastSquares,
    config: dict,
    user_items_matrix: csr_matrix,
    eval_user_ids: List[int],
    eval_actual_by_user: Dict[int, set],
    k_values: List[int],
    total_catalog_items: int,
) -> Tuple[pd.DataFrame, Dict[int, List[int]]]:
    started = time.time()
    max_k = max(k_values)

    sums = {
        k: {"precision": 0.0, "recall": 0.0, "map": 0.0, "ndcg": 0.0, "mrr": 0.0, "hitrate": 0.0}
        for k in k_values
    }
    unique_recommended_items_by_k = {k: set() for k in k_values}
    sample_recommendations_by_user = {}
    scored_users = 0

    for user_id, recommended in batch_recommend_als(
        model=model,
        user_items_matrix=user_items_matrix,
        user_ids=eval_user_ids,
        k=max_k,
        batch_size=RECOMMEND_BATCH_SIZE,
    ):
        actual = eval_actual_by_user[user_id]
        if len(sample_recommendations_by_user) < 10:
            sample_recommendations_by_user[user_id] = recommended
        update_metric_sums(recommended, actual, k_values, sums, unique_recommended_items_by_k)
        scored_users += 1

    if scored_users == 0:
        raise ValueError(f"No users scored for {config['model_name']}.")

    rows = []
    elapsed_seconds = time.time() - started

    for k in k_values:
        row = {
            "model_name": config["model_name"],
            "stage": config.get("stage", "unknown"),
            "eval_split": EVAL_SPLIT_NAME,
            "warm_start_only": True,
            "k": int(k),
            "n_eval_users": int(scored_users),
            "factors": int(config["factors"]),
            "alpha": float(config["alpha"]),
            "regularization": float(config["regularization"]),
            "iterations": int(config["iterations"]),
            "precision_at_k": sums[k]["precision"] / scored_users,
            "recall_at_k": sums[k]["recall"] / scored_users,
            "map_at_k": sums[k]["map"] / scored_users,
            "ndcg_at_k": sums[k]["ndcg"] / scored_users,
            "mrr_at_k": sums[k]["mrr"] / scored_users,
            "hitrate_at_k": sums[k]["hitrate"] / scored_users,
            "unique_recommended_items": int(len(unique_recommended_items_by_k[k])),
            "catalog_coverage": len(unique_recommended_items_by_k[k]) / total_catalog_items if total_catalog_items > 0 else np.nan,
            "eval_elapsed_seconds": round(elapsed_seconds, 2),
            "max_eval_users_per_split": MAX_EVAL_USERS_PER_SPLIT,
        }
        rows.append(row)

    return pd.DataFrame(rows), sample_recommendations_by_user


In [ ]:
def get_primary_row(metrics_df: pd.DataFrame) -> dict:
    primary = metrics_df[metrics_df["k"] == PRIMARY_K].copy()
    if len(primary) != 1:
        raise ValueError("Expected exactly one primary K row.")
    return primary.iloc[0].to_dict()


def is_better_primary_row(candidate: dict, incumbent: Optional[dict]) -> bool:
    if incumbent is None:
        return True

    sort_keys = ["recall_at_k", "ndcg_at_k", "mrr_at_k", "map_at_k", "catalog_coverage"]
    candidate_tuple = tuple(float(candidate[key]) for key in sort_keys)
    incumbent_tuple = tuple(float(incumbent[key]) for key in sort_keys)
    return candidate_tuple > incumbent_tuple


def save_best_model_artifacts(model: AlternatingLeastSquares, config: dict, primary_row: dict) -> None:
    if not SAVE_BEST_MODEL_FACTORS:
        return

    BEST_MODEL_DIR.mkdir(parents=True, exist_ok=True)

    np.save(BEST_MODEL_DIR / "user_factors.npy", model.user_factors)
    np.save(BEST_MODEL_DIR / "item_factors.npy", model.item_factors)

    if SAVE_BEST_MODEL_PICKLE:
        safe_pickle(model, BEST_MODEL_DIR / "als_model.pkl")

    save_json(config, BEST_MODEL_DIR / "best_als_tuning_config.json")
    save_json(primary_row, BEST_MODEL_DIR / "best_als_tuning_primary_row.json")

    print("Saved current best model factors to:", BEST_MODEL_DIR)


## 8. Run main ALS hyperparameter grid

This trains and evaluates the 18 main configurations.

The main comparison row is:

- `eval_split = validation`
- `warm_start_only = True`
- `k = 20`


In [ ]:
all_metric_frames = []
all_run_summaries = []
all_sample_recommendations = {}

best_primary_row = None
best_config = None

total_catalog_items = int(train_interactions["article_idx"].nunique())

for run_number, config in enumerate(MAIN_GRID_CONFIGS, start=1):
    print()
    print(f"Starting main grid run {run_number}/{len(MAIN_GRID_CONFIGS)}")

    started_total = time.time()
    model, train_elapsed = train_als_model(config, user_items)

    metrics_df, sample_recs = evaluate_als_on_fixed_validation_sample(
        model=model,
        config=config,
        user_items_matrix=user_items,
        eval_user_ids=tuning_eval_user_ids,
        eval_actual_by_user=val_actual_by_user,
        k_values=TOP_K_VALUES,
        total_catalog_items=total_catalog_items,
    )

    primary_row = get_primary_row(metrics_df)
    total_elapsed = time.time() - started_total

    metrics_df["train_elapsed_seconds"] = round(train_elapsed, 2)
    metrics_df["total_elapsed_seconds"] = round(total_elapsed, 2)

    all_metric_frames.append(metrics_df)
    all_sample_recommendations[config["model_name"]] = sample_recs

    run_summary = {
        **config,
        "train_elapsed_seconds": round(train_elapsed, 2),
        "total_elapsed_seconds": round(total_elapsed, 2),
        "primary_recall_at_20": float(primary_row["recall_at_k"]),
        "primary_ndcg_at_20": float(primary_row["ndcg_at_k"]),
        "primary_mrr_at_20": float(primary_row["mrr_at_k"]),
        "primary_map_at_20": float(primary_row["map_at_k"]),
        "primary_catalog_coverage_at_20": float(primary_row["catalog_coverage"]),
    }
    all_run_summaries.append(run_summary)

    display(metrics_df)

    if is_better_primary_row(primary_row, best_primary_row):
        print("New best model found:", config["model_name"])
        best_primary_row = primary_row
        best_config = config.copy()
        save_best_model_artifacts(model, best_config, best_primary_row)

    # Save partial results after every model so Kaggle output is still useful if runtime stops.
    partial_metrics = pd.concat(all_metric_frames, ignore_index=True)
    save_csv(partial_metrics, TUNING_ARTIFACTS_DIR / "als_hyperparameter_tuning_metrics_partial.csv")
    save_json({"best_config_so_far": best_config, "best_primary_row_so_far": best_primary_row}, TUNING_ARTIFACTS_DIR / "best_tuning_so_far.json")

    del model
    gc.collect()

main_grid_metrics = pd.concat(all_metric_frames, ignore_index=True)
main_grid_run_summary = pd.DataFrame(all_run_summaries)

metric_cols = [
    "precision_at_k",
    "recall_at_k",
    "map_at_k",
    "ndcg_at_k",
    "mrr_at_k",
    "hitrate_at_k",
    "catalog_coverage",
]
main_grid_metrics[metric_cols] = main_grid_metrics[metric_cols].round(6)

save_csv(main_grid_metrics, TUNING_ARTIFACTS_DIR / "main_grid_als_tuning_metrics.csv")
save_csv(main_grid_run_summary, TUNING_ARTIFACTS_DIR / "main_grid_als_tuning_run_summary.csv")

print("Main grid complete.")


## 9. Main-grid leaderboard


In [ ]:
main_grid_leaderboard = (
    main_grid_metrics[main_grid_metrics["k"] == PRIMARY_K]
    .sort_values(["recall_at_k", "ndcg_at_k", "mrr_at_k", "map_at_k", "catalog_coverage"], ascending=False)
    .reset_index(drop=True)
)

print("Main-grid leaderboard at K=20:")
display(main_grid_leaderboard)

save_csv(main_grid_leaderboard, TUNING_ARTIFACTS_DIR / "main_grid_als_tuning_leaderboard_k20.csv")


## 10. Optional focused 256-factor stage

This stage uses the top alpha/regularization combinations from the main grid and tests them with `256` factors.

This is safer than running every 256-factor combination blindly.


In [ ]:
focused_256_metrics = pd.DataFrame()
focused_256_run_summary = pd.DataFrame()

if RUN_FOCUSED_256_STAGE:
    top_combos = (
        main_grid_leaderboard[["alpha", "regularization", "iterations"]]
        .drop_duplicates()
        .head(N_TOP_COMBOS_FOR_256)
    )

    focused_256_configs = []
    for row in top_combos.itertuples(index=False):
        config = {
            "stage": "focused_256",
            "factors": int(FOCUSED_256_FACTORS),
            "alpha": float(row.alpha),
            "regularization": float(row.regularization),
            "iterations": int(row.iterations),
        }
        config["model_name"] = make_model_name(config)
        focused_256_configs.append(config)

    print("Focused 256 configs:", len(focused_256_configs))
    for config in focused_256_configs:
        print(config)

    focused_metric_frames = []
    focused_run_summaries = []

    for run_number, config in enumerate(focused_256_configs, start=1):
        print()
        print(f"Starting focused 256 run {run_number}/{len(focused_256_configs)}")

        started_total = time.time()
        model, train_elapsed = train_als_model(config, user_items)

        metrics_df, sample_recs = evaluate_als_on_fixed_validation_sample(
            model=model,
            config=config,
            user_items_matrix=user_items,
            eval_user_ids=tuning_eval_user_ids,
            eval_actual_by_user=val_actual_by_user,
            k_values=TOP_K_VALUES,
            total_catalog_items=total_catalog_items,
        )

        primary_row = get_primary_row(metrics_df)
        total_elapsed = time.time() - started_total

        metrics_df["train_elapsed_seconds"] = round(train_elapsed, 2)
        metrics_df["total_elapsed_seconds"] = round(total_elapsed, 2)

        focused_metric_frames.append(metrics_df)
        all_metric_frames.append(metrics_df)
        all_sample_recommendations[config["model_name"]] = sample_recs

        run_summary = {
            **config,
            "train_elapsed_seconds": round(train_elapsed, 2),
            "total_elapsed_seconds": round(total_elapsed, 2),
            "primary_recall_at_20": float(primary_row["recall_at_k"]),
            "primary_ndcg_at_20": float(primary_row["ndcg_at_k"]),
            "primary_mrr_at_20": float(primary_row["mrr_at_k"]),
            "primary_map_at_20": float(primary_row["map_at_k"]),
            "primary_catalog_coverage_at_20": float(primary_row["catalog_coverage"]),
        }
        focused_run_summaries.append(run_summary)

        display(metrics_df)

        if is_better_primary_row(primary_row, best_primary_row):
            print("New best model found:", config["model_name"])
            best_primary_row = primary_row
            best_config = config.copy()
            save_best_model_artifacts(model, best_config, best_primary_row)

        partial_all_metrics = pd.concat(all_metric_frames, ignore_index=True)
        save_csv(partial_all_metrics, TUNING_ARTIFACTS_DIR / "als_hyperparameter_tuning_metrics_partial.csv")
        save_json({"best_config_so_far": best_config, "best_primary_row_so_far": best_primary_row}, TUNING_ARTIFACTS_DIR / "best_tuning_so_far.json")

        del model
        gc.collect()

    if focused_metric_frames:
        focused_256_metrics = pd.concat(focused_metric_frames, ignore_index=True)
        focused_256_run_summary = pd.DataFrame(focused_run_summaries)
        focused_256_metrics[metric_cols] = focused_256_metrics[metric_cols].round(6)
        save_csv(focused_256_metrics, TUNING_ARTIFACTS_DIR / "focused_256_als_tuning_metrics.csv")
        save_csv(focused_256_run_summary, TUNING_ARTIFACTS_DIR / "focused_256_als_tuning_run_summary.csv")
else:
    print("Focused 256 stage skipped. Set RUN_FOCUSED_256_STAGE = True to run it.")


## 11. Final ALS tuning leaderboard


In [ ]:
all_tuning_metrics = pd.concat(all_metric_frames, ignore_index=True)
all_tuning_metrics[metric_cols] = all_tuning_metrics[metric_cols].round(6)

als_tuning_leaderboard = (
    all_tuning_metrics[all_tuning_metrics["k"] == PRIMARY_K]
    .sort_values(["recall_at_k", "ndcg_at_k", "mrr_at_k", "map_at_k", "catalog_coverage"], ascending=False)
    .reset_index(drop=True)
)

print("Final ALS tuning leaderboard at K=20:")
display(als_tuning_leaderboard)

save_csv(all_tuning_metrics, TUNING_ARTIFACTS_DIR / "als_hyperparameter_tuning_metrics.csv")
save_csv(als_tuning_leaderboard, TUNING_ARTIFACTS_DIR / "als_hyperparameter_tuning_leaderboard_k20.csv")

best_tuning_row = als_tuning_leaderboard.iloc[0].to_dict()
best_tuning_config = {
    "model_name": best_tuning_row["model_name"],
    "stage": best_tuning_row["stage"],
    "factors": int(best_tuning_row["factors"]),
    "alpha": float(best_tuning_row["alpha"]),
    "regularization": float(best_tuning_row["regularization"]),
    "iterations": int(best_tuning_row["iterations"]),
}

save_json(best_tuning_config, TUNING_ARTIFACTS_DIR / "best_als_hyperparameters.json")
save_json(best_tuning_row, TUNING_ARTIFACTS_DIR / "best_als_hyperparameter_row.json")

print("Best tuned ALS config:")
print(json.dumps(best_tuning_config, indent=2))


## 12. Compare tuned ALS against popularity baseline

This section answers the main question: did tuned ALS beat the popularity baseline on validation warm-start K=20?


In [ ]:
comparison_rows = []

if best_popularity_row is not None:
    comparison_rows.append({
        "model_family": "best_popularity_baseline",
        "model_name": best_popularity_row.get("model_name"),
        "k": int(best_popularity_row.get("k")),
        "precision_at_k": float(best_popularity_row.get("precision_at_k")),
        "recall_at_k": float(best_popularity_row.get("recall_at_k")),
        "map_at_k": float(best_popularity_row.get("map_at_k")),
        "ndcg_at_k": float(best_popularity_row.get("ndcg_at_k")),
        "mrr_at_k": float(best_popularity_row.get("mrr_at_k")),
        "hitrate_at_k": float(best_popularity_row.get("hitrate_at_k")),
        "catalog_coverage": float(best_popularity_row.get("catalog_coverage")),
    })

comparison_rows.append({
    "model_family": "best_tuned_als",
    "model_name": best_tuning_row.get("model_name"),
    "k": int(best_tuning_row.get("k")),
    "precision_at_k": float(best_tuning_row.get("precision_at_k")),
    "recall_at_k": float(best_tuning_row.get("recall_at_k")),
    "map_at_k": float(best_tuning_row.get("map_at_k")),
    "ndcg_at_k": float(best_tuning_row.get("ndcg_at_k")),
    "mrr_at_k": float(best_tuning_row.get("mrr_at_k")),
    "hitrate_at_k": float(best_tuning_row.get("hitrate_at_k")),
    "catalog_coverage": float(best_tuning_row.get("catalog_coverage")),
})

baseline_comparison = pd.DataFrame(comparison_rows)
display(baseline_comparison)
save_csv(baseline_comparison, TUNING_ARTIFACTS_DIR / "best_tuned_als_vs_popularity_comparison.csv")

if best_popularity_row is not None:
    decision_summary = {
        "best_tuned_als_model": best_tuning_row.get("model_name"),
        "best_popularity_model": best_popularity_row.get("model_name"),
        "als_beats_popularity_recall_at_20": float(best_tuning_row["recall_at_k"]) > float(best_popularity_row["recall_at_k"]),
        "als_beats_popularity_ndcg_at_20": float(best_tuning_row["ndcg_at_k"]) > float(best_popularity_row["ndcg_at_k"]),
        "als_beats_popularity_mrr_at_20": float(best_tuning_row["mrr_at_k"]) > float(best_popularity_row["mrr_at_k"]),
        "als_beats_popularity_map_at_20": float(best_tuning_row["map_at_k"]) > float(best_popularity_row["map_at_k"]),
        "als_beats_popularity_coverage_at_20": float(best_tuning_row["catalog_coverage"]) > float(best_popularity_row["catalog_coverage"]),
        "next_notebook_recommendation": "Use best_als_hyperparameters.json for recent-only and time-decayed ALS.",
    }
else:
    decision_summary = {
        "best_tuned_als_model": best_tuning_row.get("model_name"),
        "best_popularity_model": None,
        "next_notebook_recommendation": "Popularity baseline metrics were not found. Use best_als_hyperparameters.json for recent-only and time-decayed ALS.",
    }

print(json.dumps(decision_summary, indent=2))
save_json(decision_summary, TUNING_ARTIFACTS_DIR / "als_tuning_decision_summary.json")


## 13. Save sample recommendations from the best tuned ALS model

This is only a small audit file. It helps us see whether recommendations look sensible. The next notebook can retrain using the best configuration.


In [ ]:
# Retrain only the best model for a small recommendation artifact.
# This makes the output easy to inspect, but the key artifact for next step is best_als_hyperparameters.json.
print("Retraining best tuned ALS model for sample recommendation output...")
best_model, best_train_elapsed = train_als_model(best_tuning_config, user_items)

best_sample_user_ids = tuning_eval_user_ids[:100]
best_sample_recommendation_rows = []
max_k = max(TOP_K_VALUES)

for user_id, recs in batch_recommend_als(
    model=best_model,
    user_items_matrix=user_items,
    user_ids=best_sample_user_ids,
    k=max_k,
    batch_size=RECOMMEND_BATCH_SIZE,
):
    for rank, article_idx in enumerate(recs, start=1):
        best_sample_recommendation_rows.append({
            "customer_idx": int(user_id),
            "article_idx": int(article_idx),
            "rank": int(rank),
            "model_name": best_tuning_config["model_name"],
        })

best_sample_recommendations = pd.DataFrame(best_sample_recommendation_rows)

if len(best_sample_recommendations) > 0:
    best_sample_recommendations = best_sample_recommendations.merge(
        article_id_map[["article_idx", "article_id"]],
        on="article_idx",
        how="left",
        validate="many_to_one",
    )

    preview_cols = [
        "article_idx",
        "article_id",
        "prod_name",
        "product_type_name",
        "product_group_name",
        "colour_group_name",
        "department_name",
        "section_name",
        "garment_group_name",
    ]
    preview_cols = [col for col in preview_cols if col in articles.columns]

    best_sample_recommendations = best_sample_recommendations.merge(
        articles[preview_cols],
        on=["article_idx", "article_id"],
        how="left",
        validate="many_to_one",
    )

print_df_info(best_sample_recommendations, "Best tuned ALS sample recommendations")
save_parquet(best_sample_recommendations, TUNING_ARTIFACTS_DIR / "best_tuned_als_sample_recommendations.parquet")

# Refresh best model factors from the retrained best model.
if SAVE_BEST_MODEL_FACTORS:
    np.save(BEST_MODEL_DIR / "user_factors.npy", best_model.user_factors)
    np.save(BEST_MODEL_DIR / "item_factors.npy", best_model.item_factors)
    save_json(best_tuning_config, BEST_MODEL_DIR / "best_als_tuning_config.json")
    save_json(best_tuning_row, BEST_MODEL_DIR / "best_als_tuning_primary_row.json")

if SAVE_BEST_MODEL_PICKLE:
    safe_pickle(best_model, BEST_MODEL_DIR / "als_model.pkl")

del best_model
gc.collect()


## 14. Final summary

Upload this executed notebook back after running it. The most important outputs for the next step are:

- `als_hyperparameter_tuning_leaderboard_k20.csv`
- `best_als_hyperparameters.json`
- `als_tuning_decision_summary.json`


In [ ]:
final_summary = {
    "notebook": "05-als-hyperparameter-tuning.ipynb",
    "purpose": "Tune normal ALS hyperparameters before recent-only and time-decayed ALS.",
    "artifact_dir": str(TUNING_ARTIFACTS_DIR),
    "best_config": best_tuning_config,
    "best_row": best_tuning_row,
    "decision_summary_file": str(TUNING_ARTIFACTS_DIR / "als_tuning_decision_summary.json"),
    "leaderboard_file": str(TUNING_ARTIFACTS_DIR / "als_hyperparameter_tuning_leaderboard_k20.csv"),
    "best_hyperparameters_file": str(TUNING_ARTIFACTS_DIR / "best_als_hyperparameters.json"),
    "saved_files": sorted([p.name for p in TUNING_ARTIFACTS_DIR.glob("*")]),
}

print(json.dumps(final_summary, indent=2))

print()
print("ALS hyperparameter tuning completed successfully.")
print("Next step: upload this executed notebook/output, then build the time-decayed ALS notebook using the best hyperparameters.")
